# Task 3: RAG Pipeline Construction and Evaluation

- Objective

-Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline using the pre-built FAISS vector store and evaluate its effectiveness via qualitative analysis. This notebook consumes the production modules implemented in src/ and utils/ and demonstrates correct wiring, execution, and evaluation per rubric.

In [1]:
# Task 3: RAG Pipeline Construction and Evaluation

from typing import List, Dict, Any
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import sys
from pathlib import Path

import faiss
import pandas as pd
import pickle

# Ensure project root is on sys.path so local packages import in notebooks
_p = Path.cwd()
for _ancestor in [_p] + list(_p.parents):
    if (_ancestor / "src").exists() and (_ancestor / "utils").exists():
        sys.path.insert(0, str(_ancestor))
        break

from utils.paths import VECTOR_STORE_DIR
from src.retriever import Retriever, dynamic_k
from src.generator import AnswerGenerator
from src.rag_pipeline import RAGPipeline
from src.evaluation import evaluate_responses

print("VECTOR_STORE_DIR:", VECTOR_STORE_DIR)

d:\Python\Week 7\Intelligent-Complaint-Analysis\env7\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VECTOR_STORE_DIR: D:\Python\Week 7\Intelligent-Complaint-Analysis\vector_store


## Load Vector Store

In [2]:
# --- Load Vector Store ---
index_path = os.path.join(VECTOR_STORE_DIR, "index.faiss")
metadata_path = os.path.join(VECTOR_STORE_DIR, "metadata.pkl")

index = faiss.read_index(index_path)

with open(metadata_path, "rb") as f:
    metadata = pickle.load(f)

print("FAISS vectors:", index.ntotal)
print("Metadata items:", len(metadata))
print("Metadata sample keys:", metadata[0].keys())

FAISS vectors: 53147
Metadata items: 53147
Metadata sample keys: dict_keys(['complaint_id', 'product', 'doc_id', 'chunk_text'])


## Build retriever + generator + pipeline

In [3]:
# --- Build retriever + generator + pipeline ---
retriever = Retriever(
    index=index,
    metadata=metadata,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

generator = AnswerGenerator(
    model_name="google/flan-t5-base",   # upgraded from small
    device=-1,                         # CPU
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    generate_kwargs={
        # These are generally helpful; if your transformers version rejects them,
        # remove them and rerun (core pipeline still works).
        "repetition_penalty": 1.1,
        "no_repeat_ngram_size": 3,
    }
)

# Default k lowered; dynamic_k used for aggregate questions below
rag = RAGPipeline(retriever=retriever, generator=generator,
                  k=3, max_context_chars=900)

Device set to use cpu


##  Single question test- Sanity Check

In [4]:
# --- Single question test (Sanity Check) ---
res = rag.answer(
    "Do complaints mention discrimination? What situations are described?",
    k=dynamic_k(
        "Do complaints mention discrimination? What situations are described?", rag.k),
    return_prompt=True
)

print(res.answer)

print("\n--- Prompt (first 1200 chars) ---")
print((res.prompt or "")[:1200])

print("\n--- Sources (preview) ---")
for s in res.sources[:2]:
    print(s.get("complaint_id"), s.get("product"),
          s.get("score"), s.get("score_type"))
    print((s.get("text") or "")[:300], "\n")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MENTIONED: Yes
EVIDENCE: In next complains you will have proofs of discrimination.

--- Prompt (first 1200 chars) ---
You are a financial analyst assistant for CrediTrust.

Rules:
- Use ONLY the information in the context excerpts.
- Do not guess or use outside knowledge.
- Output EXACTLY two lines in the required format.
- EVIDENCE must be copied from the context (a short quote, ~5–20 words).
- If the context does not support an answer, respond with:
  MENTIONED: I don't know
  EVIDENCE: I don't have enough information in the provided complaints to answer that.

Required format (exactly two lines):
MENTIONED: Yes / No / I don't know
EVIDENCE: <short quote from excerpts OR REFUSAL>

Context excerpts:
[1] (complaint_id=13135229, product=Credit card, score=0.6272) Submitted without prejudice under UCC 1-308. This complaint is made in good faith and under oath as to its truth and accuracy.
[2] (complaint_id=1591204, product=Money transfers, score=0.6289) Your work in these cases is also f

## Qualitative evaluation run (5–10 questions)

In [5]:
# --- Qualitative evaluation run (5–10 questions) ---
questions = [
    "What are the most common issues customers complain about regarding credit card charges?",
    "Do complaints mention discrimination? What situations are described?",
    "Are there recurring issues with loan payment posting delays or misapplied payments?",
    "What are customers complaining about regarding overdraft or unexpected fees?",
    "Do users report problems closing accounts or cancelling services?",
    "Are there complaints about credit reporting errors or incorrect delinquency reporting?",
    "What issues do customers report with customer service responsiveness?",
    "Do complaints mention fraud or unauthorized transactions?",
    "Do credit card complaints mention late fees being charged even when payments were on time? Provide evidence.",
    "What problems do customers face when trying to dispute charges or transactions?",
    "Are there common themes in complaints about mortgage servicing or foreclosure processes?",
    "What issues do customers report regarding identity theft protection services?",
    "Do complaints mention problems with mobile banking apps or online account access?",
    "Are there recurring complaints about billing errors or incorrect statements?",
    "What issues do customers face with rewards programs or cashback offers?",
    "Are there complaints about difficulties in reaching customer support via phone or email?",
    "What problems do customers report regarding credit limit increases or decreases?",
    "Do complaints mention issues with account freezes or holds?",
    "Are there common themes in complaints about student loan servicing?",
    "What issues do customers report with wire transfers or international transactions?",
    "Do complaints mention problems with automatic payments or recurring billing?",
    "Are there recurring complaints about account security or data breaches?",
    "What issues do customers face with payment processing delays?",
    "Do complaints mention difficulties in understanding terms and conditions?",
    "What problems do customers report regarding financial hardship programs?",
    "Are there complaints about difficulties in disputing fraudulent charges?",
    "What issues do customers report with account statements or transaction histories?",
    "Do complaints mention problems with customer loyalty programs or perks?",
    "Are there common themes in complaints about credit counseling services?",
    "What issues do customers face with overdraft protection services?",
    "Do complaints mention problems with account verification or identity checks?",
    "What problems do customers report regarding late payment notifications?",
    
]

# (Optional but recommended) quick sanity check that you reloaded the patched pipeline
# If your RAGPipeline has these attributes, you're on the updated version.
print("aggregate_generate_kwargs:", getattr(
    rag, "aggregate_generate_kwargs", None))

rag_results = []
for q in questions:
    # If you kept my updated rag_pipeline.py, it can auto-apply dynamic_k when k=None.
    # But leaving your explicit dynamic_k call is perfectly fine.
    kq = dynamic_k(q, base_k=rag.k)
    rag_results.append(rag.answer(q, k=kq, return_prompt=False))

results_for_eval: List[Dict[str, Any]] = [
    {"question": r.question, "answer": r.answer, "sources": r.sources}
    for r in rag_results
]

df = evaluate_responses(results_for_eval)

df_eval = df.copy()
df_eval["qualitative_feedback"] = ""

display(df_eval)
# df_eval.to_csv("processed/task3_eval.csv", index=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (2438 > 512). Running this sequence through the model will result in indexing errors


aggregate_generate_kwargs: {'max_new_tokens': 240, 'num_beams': 4, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3}


,Question,Generated Answer,MENTIONED (parsed),EVIDENCE (parsed),ISSUES (parsed),NUM_BULLETS (parsed),Retrieved Sources (show 1-2),Quality Score (1-5),Comments/Analysis,qualitative_feedback
0,What are the most common issues customers comp...,"ISSUES:\n- 6064) However, I am filing this com...",,,"- 6064) However, I am filing this complaint be...",7,complaint_id=8235813 | product=Credit card | d...,TBD,TBD (manual review),
1,Do complaints mention discrimination? What sit...,MENTIONED: Yes\nEVIDENCE: In next complains yo...,Yes,In next complains you will have proofs of disc...,,0,complaint_id=13135229 | product=Credit card | ...,TBD,TBD (manual review),
2,Are there recurring issues with loan payment p...,"MENTIONED: Yes\nEVIDENCE: Additionally, the le...",Yes,"Additionally, the lender claimed that the dela...",,0,complaint_id=7176375 | product=Credit card | d...,TBD,TBD (manual review),
3,What are customers complaining about regarding...,ISSUES:\n- 5691) Overdraft fees was taken out ...,,,- 5691) Overdraft fees was taken out multiple ...,7,complaint_id=7492299 | product=Savings account...,TBD,TBD (manual review),
4,Do users report problems closing accounts or c...,MENTIONED: Yes\nEVIDENCE: notified in any form...,Yes,notified in any form of my account being close...,,0,complaint_id=2757890 | product=Credit card | d...,TBD,TBD (manual review),
5,Are there complaints about credit reporting er...,MENTIONED: Yes\nEVIDENCE: [3] (complaint_id=82...,Yes,"[3] (complaint_id=8218564, product=Credit card...",,0,complaint_id=7701837 | product=Credit card | d...,TBD,TBD (manual review),
6,What issues do customers report with customer ...,ISSUES:\n- 6046) I am reading that many custom...,,,- 6046) I am reading that many customers are h...,7,complaint_id=8705243 | product=Money transfers...,TBD,TBD (manual review),
7,Do complaints mention fraud or unauthorized tr...,MENTIONED: Yes\nEVIDENCE: [2] (complaint_id=12...,Yes,"[2] (complaint_id=12360564, product=Savings ac...",,0,complaint_id=13732011 | product=Credit card | ...,TBD,TBD (manual review),
8,Do credit card complaints mention late fees be...,MENTIONED: Yes\nEVIDENCE: methods of finance c...,Yes,methods of finance charges and late fees.,,0,complaint_id=2765793 | product=Credit card | d...,TBD,TBD (manual review),
9,What problems do customers face when trying to...,ISSUES:\n- 5761) Their policies via https : //...,,,- 5761) Their policies via https : //www | - 5...,3,complaint_id=13123277 | product=Credit card | ...,TBD,TBD (manual review),


## Export Markdown table for your report

In [6]:
# --- Export Markdown table for your report ---
print(df.to_markdown(index=False))
# save csv for scoring
df.to_csv(r"D:\Python\Week 7\Intelligent-Complaint-Analysis\data\processed\task3_rag_qualitative_evaluation.csv", index=False)
print("Saved task3_rag_qualitative_evaluation.csv")

| Question                                                                                                     | Generated Answer                                                                                                                                                                    | MENTIONED (parsed)   | EVIDENCE (parsed)                                                                                                                                                         | ISSUES (parsed)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

## Load and preview the saved qualitative evaluation results

In [7]:
# Load and preview the saved qualitative evaluation results
import pandas as pd
df_loaded = pd.read_csv(r"D:\Python\Week 7\Intelligent-Complaint-Analysis\data\processed\task3_rag_qualitative_evaluation.csv")
print('Loaded', len(df_loaded), 'rows from task3_rag_qualitative_evaluation.csv')
print(df_loaded.to_markdown(index=False))

Loaded 32 rows from task3_rag_qualitative_evaluation.csv
| Question                                                                                                     | Generated Answer                                                                                                                                                                    | MENTIONED (parsed)   | EVIDENCE (parsed)                                                                                                                                                         | ISSUES (parsed)                                                                                                                                                                                                                                                                                                                                                                                                                                                      

## Evaluation Analysis

